In [1]:
import numpy as np

def load_glove(filepath):
    words = []
    vectors = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            words.append(parts[0])
            vectors.append([float(x) for x in parts[1:]])
    word2idx = {w: i for i, w in enumerate(words)}
    vectors = np.array(vectors, dtype=np.float32)
    return words, vectors, word2idx

def cossim(a, b):
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return np.dot(a, b) / (norm_a * norm_b)

def analogy(w1, w2, w3, words, vectors, word2idx, top_k=5):
    for w in [w1, w2, w3]:
        if w not in word2idx:
            return []

    v1 = vectors[word2idx[w1]]
    v2 = vectors[word2idx[w2]]
    v3 = vectors[word2idx[w3]]

    target_vector = v1 - v2 + v3

    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    normalized_vectors = vectors / (norms + 1e-8)
    
    target_norm = np.linalg.norm(target_vector)
    normalized_target = target_vector / (target_norm + 1e-8)

    similarities = np.dot(normalized_vectors, normalized_target)

    exclude_indices = {word2idx[w1], word2idx[w2], word2idx[w3]}
    sorted_indices = np.argsort(similarities)[::-1]

    results = []
    for idx in sorted_indices:
        if idx not in exclude_indices:
            results.append((words[idx], similarities[idx]))
            if len(results) >= top_k:
                break

    return results

if __name__ == "__main__":
    words, vectors, word2idx = load_glove("vectors_skipgram.txt")

    results = analogy("king", "man", "woman", words, vectors, word2idx, top_k=5)
    print("king - man + woman:")
    for word, score in results:
        print(f"  {word}: {score:.4f}")

    results = analogy("paris", "france", "germany", words, vectors, word2idx, top_k=5)
    print("\nparis - france + germany:")
    for word, score in results:
        print(f"  {word}: {score:.4f}")

king - man + woman:
  son: 0.4176
  queen: 0.3927
  kings: 0.3849
  daughter: 0.3748
  birth: 0.3675

paris - france + germany:
  berlin: 0.4858
  munich: 0.4814
  vienna: 0.3863
  warsaw: 0.3754
  german: 0.3638
